In [ ]:
import subprocess, sys
 
def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
 
install("unsloth[colab-new]")   # Unsloth avec support Flash Attention
install("trl")                  # Trainer HuggingFace pour fine-tuning
install("peft")                 # PEFT = Parameter-Efficient Fine-Tuning (LoRA)
install("bitsandbytes")         # Quantification 4-bit pour économiser la VRAM
 
print("Dépendances installées.")

In [ ]:
from kaggle_secrets import UserSecretsClient
HF_TOKEN = UserSecretsClient().get_secret("token_medgemma")

from huggingface_hub import login
login(HF_TOKEN)

from unsloth import FastVisionModel   # VisionModel car MedGemma est multimodal
import torch
 
MODEL_ID = "google/medgemma-4b-it"

In [ ]:
import json
import pandas as pd
import numpy as np
from pathlib import Path
from PIL import Image
 
# --- Localisation du dataset CheXpert ---
DATA_ROOT = next(p.parent for p in Path("/kaggle/input").rglob("train.csv"))
CSV_PATH  = DATA_ROOT / "train.csv"
print("DATA_ROOT =", DATA_ROOT)
 
OPACITY_COLS = ["Lung Opacity", "Consolidation", "Pneumonia", "Edema", "Atelectasis"]
 
# --- Prompt d'entraînement ---
# On utilise le prompt amélioré (PROMPT_IMPROVED) comme instruction de base.
# Le modèle apprendra à toujours produire le JSON demandé dans ce format.
TRAIN_PROMPT = """You are an educational radiology assistant for engineering students.
You are not a clinician and must not provide a definitive diagnosis.
Analyze this frontal chest X-ray and classify it as: normal, suspected_opacity, or uncertain.
Return ONLY valid JSON with this exact schema:
{
  "image_quality": "good | limited | poor",
  "predicted_class": "normal | suspected_opacity | uncertain",
  "confidence": 0.0,
  "visual_evidence": ["observation 1", "observation 2"],
  "justification": "2-4 cautious sentences describing findings and reasoning",
  "limitations": ["limitation 1"],
  "warning": "Educational prototype only. Not for diagnosis."
}
Rules:
- confidence < 0.50 forces predicted_class to "uncertain"
- Never invent patient history. No diagnosis language.
- Return ONLY the JSON object, nothing else."""
 
 
def ground_truth_label(row):
    """Mappe les labels CheXpert vers nos 3 classes."""
    if row["No Finding"] == 1.0:
        return "normal"
    if any(row.get(c) == 1.0 for c in OPACITY_COLS):
        return "suspected_opacity"
    return None   # cas ambigu : exclu de l'entraînement
 
 
def image_path_from_row(row):
    rel = row["Path"].split("/", 1)[1]
    return DATA_ROOT / rel
 
 
import random

TEMPLATES_NORMAL = [
    {
        "confidence": 0.80,
        "visual_evidence": ["Lung fields appear clear bilaterally", "No obvious consolidation or opacity detected", "Costophrenic angles appear sharp", "Cardiac silhouette within normal limits"],
        "justification": "The chest radiograph demonstrates clear lung fields without obvious areas of consolidation, infiltrate or opacity. The cardiomediastinal silhouette appears unremarkable. This is an educational assessment and should not replace clinical evaluation.",
    },
    {
        "confidence": 0.85,
        "visual_evidence": ["Well-aerated lung fields", "No focal consolidation identified", "Normal cardiothoracic ratio", "Sharp costophrenic recesses bilaterally"],
        "justification": "This frontal chest X-ray shows well-expanded, clear lungs without evidence of focal opacity or consolidation. Heart size appears within normal range. This educational assessment does not substitute for professional radiological review.",
    },
    {
        "confidence": 0.78,
        "visual_evidence": ["Symmetric lung expansion", "No areas of increased density noted", "Clear costophrenic angles", "Unremarkable mediastinal contour"],
        "justification": "The radiograph reveals symmetric, well-inflated lungs with no visible opacity, infiltrate, or consolidation. Mediastinal structures appear normal in position and size. Educational interpretation only.",
    },
]

TEMPLATES_OPACITY = [
    {
        "confidence": 0.72,
        "visual_evidence": ["Increased density or opacity visible in lung field(s)", "Possible consolidation pattern detected", "Asymmetric lung markings observed"],
        "justification": "The radiograph shows increased density in one or more lung zones, which may suggest opacity, consolidation or infiltrate. Clinical correlation and further evaluation by a radiologist are strongly recommended. This is an educational prototype and cannot provide a definitive diagnosis.",
    },
    {
        "confidence": 0.68,
        "visual_evidence": ["Focal area of increased opacity noted", "Possible airspace disease pattern", "Loss of normal lung markings in affected region"],
        "justification": "There is a region of increased radiographic density that may represent consolidation, infiltrate or atelectasis. These findings warrant clinical correlation and radiologist review. This educational tool cannot establish a diagnosis.",
    },
    {
        "confidence": 0.75,
        "visual_evidence": ["Patchy opacification observed", "Possible consolidative changes", "Irregular lung field density"],
        "justification": "The chest radiograph demonstrates patchy areas of increased opacity, potentially indicating consolidation or infiltrative process. Professional radiological evaluation is strongly advised. This is an educational assessment only.",
    },
]

def build_training_json(gt_label, image_quality="good"):
    pool = TEMPLATES_NORMAL if gt_label == "normal" else TEMPLATES_OPACITY
    chosen = random.choice(pool)   # <-- variabilité : plus un seul bloc figé
    return {
        "image_quality": image_quality,
        "predicted_class": gt_label,
        "confidence": chosen["confidence"],
        "visual_evidence": chosen["visual_evidence"],
        "justification": chosen["justification"],
        "limitations": ["Educational prototype — not validated for clinical use", "Subtle findings may be missed by automated analysis"],
        "warning": "Educational prototype only. Not for diagnosis."
    }

 
 
def load_and_preprocess_image(image_path, target_size=(384, 384)):
    """Charge et redimensionne une image pour l'entraînement."""
    img = Image.open(image_path).convert("RGB")
    img = img.resize(target_size, Image.LANCZOS)
    return img
 
 
def build_training_sample(row):
    """
    Construit un exemple d'entraînement au format chat template.
    Retourne un dict {messages, image} compatible avec Unsloth/TRL.
    """
    gt = row["gt"]
    img_path = image_path_from_row(row)
 
    try:
        image = load_and_preprocess_image(img_path)
    except Exception as e:
        return None   # image corrompue ou manquante : on saute
 
    target_json = build_training_json(gt)
 
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text",  "text": TRAIN_PROMPT}
            ]
        },
        {
            "role": "assistant",
            "content": [
                {"type": "text", "text": json.dumps(target_json, indent=2)}
            ]
        }
    ]
 
    return {"messages": messages}
 
 
# --- Construction du dataset ---
# On tire un échantillon équilibré entre "normal" et "suspected_opacity"
# pour éviter le biais de classe (le modèle apprendrait sinon à toujours
# prédire la classe majoritaire).
 
df = pd.read_csv(CSV_PATH)
df = df[df["Frontal/Lateral"] == "Frontal"].copy()
df["gt"] = df.apply(ground_truth_label, axis=1)
df = df[df["gt"].notna()]
 
N_PER_CLASS = 25   # 120 normal + 120 suspected_opacity = 240 exemples
VAL_SPLIT   = 0.15  # 15% pour la validation (36 exemples)
 
df_normal   = df[df["gt"] == "normal"].sample(n=N_PER_CLASS, random_state=42)
df_opacity  = df[df["gt"] == "suspected_opacity"].sample(n=N_PER_CLASS, random_state=42)
df_train_full = pd.concat([df_normal, df_opacity]).sample(frac=1, random_state=42)
 
n_val   = int(len(df_train_full) * VAL_SPLIT)
df_val  = df_train_full.iloc[:n_val]
df_train = df_train_full.iloc[n_val:]
 
print(f"Exemples d'entraînement : {len(df_train)}")
print(f"Exemples de validation  : {len(df_val)}")
 
# Conversion en liste de dicts
train_samples = [s for row in df_train.itertuples() for s in [build_training_sample(row._asdict())] if s]
val_samples   = [s for row in df_val.itertuples()   for s in [build_training_sample(row._asdict())] if s]
 
print(f"✅ Dataset construit : {len(train_samples)} train / {len(val_samples)} val")

In [ ]:
model, tokenizer = FastVisionModel.from_pretrained(
    MODEL_ID,
    max_seq_length=2048,
    load_in_4bit=True,
    dtype=None,   # auto-détection (bfloat16 si disponible, sinon float16)
)

model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=False,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=False,      # CHANGÉ : False au lieu de True — évite d'écraser le raisonnement
    r=4,                              # CHANGÉ : 4 au lieu de 8 — moins de capacité à "écraser" le comportement de base
    lora_alpha=8,                     # CHANGÉ : 8 au lieu de 16 — suit la convention alpha = 2*r
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
 
print("✅ Modèle chargé avec LoRA configuré.")
print(f"   Paramètres entraînables : {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
print(f"   Paramètres totaux       : {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
from datasets import Dataset
 
train_dataset = Dataset.from_list(train_samples)
val_dataset   = Dataset.from_list(val_samples)
 
print(train_dataset)
print("Exemple d'entraînement :")
print(train_dataset[0]["messages"][1]["content"][0]["text"][:200], "...")   # début du JSON cible

In [ ]:
from unsloth import is_bfloat16_supported
 
def formatting_func(examples):
    """
    Gère à la fois :
    - un seul exemple : {"messages": [msg_user, msg_assistant]}
    - un batch        : {"messages": [[msg_user, msg_assistant], [msg_user, msg_assistant], ...]}
    Renvoie TOUJOURS une liste de strings (exigé par Unsloth/TRL).
    """
    messages_field = examples["messages"]

    # Cas batch : liste de conversations (chaque conversation = liste de messages)
    if isinstance(messages_field[0], dict):
        # un seul exemple a été passé (pas un batch) -> on l'enveloppe
        conversations = [messages_field]
    else:
        conversations = messages_field

    texts = [
        tokenizer.apply_chat_template(conv, tokenize=False, add_generation_prompt=False)
        for conv in conversations
    ]
    return texts

In [ ]:
from trl import SFTTrainer, SFTConfig
from transformers import TrainingArguments
 
# Active le mode entraînement d'Unsloth (optimisations spécifiques)
FastVisionModel.for_training(model)
 
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=None,   # Unsloth gère la collation nativement
    formatting_func=formatting_func,
    args=SFTConfig(
        output_dir="./lora_medgemma_cxr",       # dossier de sauvegarde des checkpoints
        num_train_epochs=1,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,           # batch effectif = 4
        warmup_ratio=0.05,
        learning_rate=5e-5,
        fp16=not is_bfloat16_supported(),        # float16 sur T4, bfloat16 sur A100
        bf16=is_bfloat16_supported(),
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=50,
        save_strategy="steps",
        save_steps=100,
        save_total_limit=2,                      # garde seulement les 2 meilleurs checkpoints
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        optim="adamw_8bit",                      # optimizer 8-bit d'Unsloth (économise la VRAM)
        weight_decay=0.01,
        lr_scheduler_type="cosine",              # décroissance cosinus du lr
        seed=42,
        report_to="none",                        # désactive W&B / MLflow
        dataset_text_field="text",
        max_seq_length=2048,
        remove_unused_columns=False,             # important pour garder les images
    ),
)
 
print("✅ Trainer configuré.")
print(f"   Steps totaux estimés : {len(train_dataset) * 3 // 4}") 

In [ ]:
import time

print("Lancement du fine-tuning LoRA...")
start = time.time()

trainer_stats = trainer.train()

duration = time.time() - start
print(f"\nEntraînement terminé en {duration/60:.1f} minutes.")
print(f"   Loss finale (train) : {trainer_stats.training_loss:.4f}")

In [ ]:
# On sauvegarde deux choses :
#   1) Les adaptateurs LoRA seuls (légers, ~50 Mo) → suffisant pour la démo
#   2) Le modèle fusionné (LoRA mergé dans le modèle de base) → pour l'inférence

LORA_SAVE_PATH   = "./lora_medgemma_cxr_final"
MERGED_SAVE_PATH = "./medgemma_cxr_merged"

# Sauvegarde des adaptateurs LoRA uniquement
model.save_pretrained(LORA_SAVE_PATH)
tokenizer.save_pretrained(LORA_SAVE_PATH)
print(f"✅ Adaptateurs LoRA sauvegardés dans : {LORA_SAVE_PATH}")

In [ ]:
import gc

# Supprime l'optimiseur et le scheduler du trainer (les plus gros consommateurs)
if hasattr(trainer, "optimizer") and trainer.optimizer is not None:
    del trainer.optimizer
if hasattr(trainer, "lr_scheduler") and trainer.lr_scheduler is not None:
    del trainer.lr_scheduler

# Supprime l'objet trainer entier (on n'en a plus besoin, le modèle est sauvegardé)
del trainer

gc.collect()
torch.cuda.empty_cache()

print(f"VRAM allouée après nettoyage : {torch.cuda.memory_allocated()/1e9:.2f} Go")
print(f"VRAM réservée après nettoyage : {torch.cuda.memory_reserved()/1e9:.2f} Go")

In [ ]:
FastVisionModel.for_inference(model)

test_diversity = pd.concat([
    df_val[df_val["gt"] == "normal"].head(3),
    df_val[df_val["gt"] == "suspected_opacity"].head(3)
])

In [ ]:
from __future__ import annotations
import re
import json
import numpy as np
from PIL import Image

# Configuration issue du cahier des charges
ALLOWED_CLASSES = {"normal", "suspected_opacity", "uncertain"}
# NB : "warning" n'est PAS exige du VLM lui-meme. Ce champ est systematiquement
# injecte/ecrase par apply_safety_guardrails (WARNING_TEXT), qu'il soit valide ou non.
REQUIRED_KEYS = {"image_quality", "predicted_class", "confidence", "visual_evidence", "justification", "limitations"}
WARNING_TEXT = "Prototype pedagogique. Non destine au diagnostic. Validation par un professionnel qualifie requise."


def check_image_quality(image_path):
    """
    Verifie la qualite technique de la radiographie (CXR) AVANT soumission au VLM.
    Filtre de securite amont pour eviter le traitement d'images inexploitables.
    """
    img = Image.open(image_path).convert("L")  # niveaux de gris
    arr = np.array(img, dtype=np.float32)

    issues = []

    mean_brightness = arr.mean()
    if mean_brightness < 30:
        issues.append("(trop sombre)")
    elif mean_brightness > 220:
        issues.append("(trop claire)")

    std = arr.std()
    if std < 20:
        issues.append("contraste trop faible")

    w, h = img.size
    if w < 224 or h < 224:
        issues.append(f"resolution insuffisante ({w}x{h})")

    ratio = w / h
    if ratio < 0.7 or ratio > 1.4:
        issues.append(f"ratio d'aspect inhabituel ({ratio:.2f})")

    pct_black = (arr < 5).mean()
    pct_white = (arr > 250).mean()
    if pct_black > 0.5:
        issues.append(f"trop de pixels noirs ({pct_black:.0%})")
    if pct_white > 0.3:
        issues.append(f"trop de pixels satures ({pct_white:.0%})")

    if len(issues) >= 2:
        quality = "poor"
    elif issues:
        quality = "limited"
    else:
        quality = "good"

    return {"quality": quality, "issues": issues}


def validate_prediction(pred):
    """Valide la conformite structurelle et clinique du JSON produit par le VLM."""
    errors = []

    missing = REQUIRED_KEYS - set(pred)
    if missing:
        errors.append(f"missing keys: {sorted(missing)}")

    if pred.get("predicted_class") not in ALLOWED_CLASSES:
        errors.append("invalid predicted_class")

    try:
        conf = float(pred.get("confidence", -1))
        if not (0 <= conf <= 1):
            errors.append("confidence outside [0,1]")
    except (TypeError, ValueError):
        errors.append("confidence is not numeric")

    return not errors, errors


def apply_safety_guardrails(pred):
    """
    Applique le filet de securite (post-processing). Force la prudence si le
    schema est invalide ou si la qualite de l'image est degradee.
    """
    valid, errors = validate_prediction(pred)

    if not valid:
        pred["predicted_class"] = "uncertain"
        try:
            current_conf = float(pred.get("confidence", 0.0))
            pred["confidence"] = min(current_conf, 0.5)
        except (TypeError, ValueError):
            pred["confidence"] = 0.0

        current_limits = pred.get("limitations")
        if isinstance(current_limits, list):
            current_limits.append("guardrail triggered: invalid output schema")
        elif isinstance(current_limits, str):
            pred["limitations"] = [current_limits, "guardrail triggered: invalid output schema"]
        else:
            pred["limitations"] = ["guardrail triggered: invalid output schema"]

    try:
        conf_value = float(pred.get("confidence", 0.0))
    except (TypeError, ValueError):
        conf_value = 0.0

    if pred.get("image_quality") in {"limited", "poor"} and conf_value < 0.6:
        pred["predicted_class"] = "uncertain"

    pred["warning"] = WARNING_TEXT
    pred["guardrail_errors"] = errors

    return pred


CXR_LEXICON = {"poumon", "poumons", "thorax", "thoracique", "plevre", "coeur", "diaphragme", "mediastin", "lung", "chest", "opacity", "infiltrat"}


def check_hallucination_lexicon(pred):
    """Verifie si le texte genere contient au moins un mot-cle du domaine thoracique."""
    text_to_check = f"{pred.get('visual_evidence', '')} {pred.get('justification', '')}".lower()

    if pred.get("predicted_class") in {"normal", "suspected_opacity"}:
        if not any(word in text_to_check for word in CXR_LEXICON):
            return False, "Le modele semble hors-sujet ou l'image n'est pas une radiographie thoracique."

    return True, None


CRITICAL_KEYWORDS = {"grave", "foudroyant", "masse", "pneumothorax", "epanchement", "detresse"}


def check_semantic_coherence(pred):
    """Empeche le modele de classer en 'normal' si le texte contient des indices de pathologie lourde."""
    justification = str(pred.get("justification", "")).lower()
    predicted_class = pred.get("predicted_class")

    if predicted_class == "normal" and any(word in justification for word in CRITICAL_KEYWORDS):
        return False, "Incoherence semantique detectee (mots critiques trouves pour une classe 'normal')."

    return True, None


def apply_uncertainty_threshold(pred, lower_bound=0.4, upper_bound=0.65):
    """Si la confiance est dans la zone grise, force la classe 'uncertain'."""
    try:
        conf = float(pred.get("confidence", 0.0))
    except (TypeError, ValueError):
        conf = 0.0

    if lower_bound <= conf <= upper_bound:
        if pred.get("predicted_class") != "uncertain":
            pred["predicted_class"] = "uncertain"
            pred.setdefault("limitations", []).append(f"Classe modifiee en 'uncertain' : confiance dans la zone grise ({conf:.2f}).")

    return pred


def finalize_prediction(pred):
    """
    Chaine complete de garde-fous AVAL, appliquee a toute prediction (VLM ou repli
    pre-VLM) avant qu'elle ne soit retournee/loggee.
    """
    triggered = []

    pred = apply_safety_guardrails(pred)
    if pred.get("guardrail_errors"):
        triggered.append("schema_or_regulatory")

    ok_lexicon, msg_lexicon = check_hallucination_lexicon(pred)
    if not ok_lexicon:
        pred["predicted_class"] = "uncertain"
        pred.setdefault("limitations", []).append(msg_lexicon)
        triggered.append("hallucination_lexicon")

    ok_coherence, msg_coherence = check_semantic_coherence(pred)
    if not ok_coherence:
        pred["predicted_class"] = "uncertain"
        pred.setdefault("limitations", []).append(msg_coherence)
        triggered.append("semantic_coherence")

    before_class = pred.get("predicted_class")
    pred = apply_uncertainty_threshold(pred)
    if pred.get("predicted_class") != before_class:
        triggered.append("uncertainty_threshold")

    pred["warning"] = WARNING_TEXT
    pred["guardrails_triggered"] = triggered

    return pred


def parse_prediction(text):
    """
    Parse la sortie brute du VLM en dictionnaire structure conforme au schema
    REQUIRED_KEYS attendu par les garde-fous.
    """
    output = {}

    match_json = re.search(r"\{.*\}", text, re.DOTALL)
    if match_json:
        try:
            parsed = json.loads(match_json.group(0))
            if isinstance(parsed, dict):
                output.update(parsed)
        except json.JSONDecodeError:
            pass

    def _extract_string_field(key):
        pattern = rf'"{key}"\s*:\s*"(.*?)(?:"\s*,\s*"[a-zA-Z_]+"\s*:|"\s*\}}|$)'
        m = re.search(pattern, text, re.DOTALL)
        return m.group(1).strip() if m else ""

    def _extract_list_field(key):
        pattern = rf'"{key}"\s*:\s*\[(.*?)(?:\]|"[a-zA-Z_]+"\s*:|$)'
        m = re.search(pattern, text, re.DOTALL)
        if not m:
            return []
        return re.findall(r'"([^"]*)"', m.group(1))

    if "predicted_class" not in output:
        if re.search(r'predicted_class"?\s*:\s*"?normal', text):
            output["predicted_class"] = "normal"
        elif re.search(r'predicted_class"?\s*:\s*"?suspe?cted_opacity', text):
            output["predicted_class"] = "suspected_opacity"
        else:
            output["predicted_class"] = "uncertain"

    CLASS_ALIASES = {
        "unsure": "uncertain", "not sure": "uncertain", "ambiguous": "uncertain",
        "indeterminate": "uncertain", "unclear": "uncertain",
        "abnormal": "suspected_opacity", "opacity": "suspected_opacity",
        "suspicious": "suspected_opacity",
        "clear": "normal", "no finding": "normal", "no abnormality": "normal",
    }
    if output.get("predicted_class") in CLASS_ALIASES:
        output["predicted_class"] = CLASS_ALIASES[output["predicted_class"]]

    if "confidence" not in output:
        match = re.search(r'"confidence":\s*([\d.]+)', text)
        if match:
            try:
                output["confidence"] = float(match.group(1))
            except ValueError:
                output["confidence"] = 0.0
        else:
            output["confidence"] = 0.0

    try:
        conf = float(output.get("confidence", 0.0))
        if conf > 1.0:
            conf = conf / 100.0
        output["confidence"] = round(max(0.0, min(conf, 1.0)), 4)
    except (TypeError, ValueError):
        output["confidence"] = 0.0
    
    output.setdefault("image_quality", "limited")
    if not output.get("visual_evidence"):
        output["visual_evidence"] = _extract_list_field("visual_evidence")
    if not output.get("justification"):
        output["justification"] = _extract_string_field("justification")
    if not output.get("limitations"):
        output["limitations"] = _extract_list_field("limitations")
    output.setdefault("warning", "")

    output["raw"] = text
    return output

In [ ]:
import re
import sys
from pathlib import Path


print("guardrails.py (fonctions) chargées depuis la cellule précédente")


# Repasse en mode inférence
FastVisionModel.for_inference(model)

PROMPT_FINETUNED = TRAIN_PROMPT   # même prompt que l'entraînement

def predict_finetuned(image_path, target_size=(384, 384)):
    image = Image.open(image_path).convert("RGB").resize(target_size, Image.LANCZOS)
    messages = [{"role": "user", "content": [
        {"type": "image", "image": image},
        {"type": "text", "text": PROMPT_FINETUNED}
    ]}]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors="pt"
    ).to(model.device)
    input_len = inputs["input_ids"].shape[-1]
    with torch.inference_mode():
        outputs = model.generate(**inputs, max_new_tokens=400, do_sample=False, repetition_penalty=1.2, temperature=1.0)
    text = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True).strip()
    del inputs, outputs
    torch.cuda.empty_cache()
    pred = parse_prediction(text)
    pred["image_file"] = str(Path(image_path).name)
    return finalize_prediction(pred)

# --- Test rapide (une image) ---
_test_row = df_val.iloc[0]
_test_path = image_path_from_row(_test_row)
print(f"Vérité terrain : {_test_row['gt']}")
result = predict_finetuned(_test_path)
print(json.dumps({k: v for k, v in result.items() if k != "raw"}, indent=2))

# --- Test ANTI-COLLAPSE sur 6 images variées ---
test_diversity = pd.concat([
    df_val[df_val["gt"] == "normal"].head(3),
    df_val[df_val["gt"] == "suspected_opacity"].head(3)
])
preds_diversity = [predict_finetuned(image_path_from_row(row))["predicted_class"] for _, row in test_diversity.iterrows()]
print("\nPrédictions test diversité :", preds_diversity)
print("Classes distinctes utilisées :", set(preds_diversity))
if len(set(preds_diversity)) == 1:
    print("ALERTE : mode collapse détecté ! Le modèle prédit une seule classe.")
else:
    print("Pas de collapse détecté, on continue.")

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score, confusion_matrix
import warnings, re, json, gc, time
warnings.filterwarnings("ignore")

# --- Jeu de test (40 images, hors train/val) ---
df_test_normal  = df[df["gt"] == "normal"].drop(index=df_train_full.index, errors="ignore").sample(n=20, random_state=99)
df_test_opacity = df[df["gt"] == "suspected_opacity"].drop(index=df_train_full.index, errors="ignore").sample(n=20, random_state=99)
df_test = pd.concat([df_test_normal, df_test_opacity]).sample(frac=1, random_state=99)
print(f"Jeu de test : {len(df_test)} images ({(df_test['gt']=='normal').sum()} normal, {(df_test['gt']=='suspected_opacity').sum()} opacity)")


def evaluate_model(predict_fn, sample, model_name="model", checkpoint_path=None):
    results = []
    start_time = time.time()

    for i, (_, row) in enumerate(sample.iterrows()):
        path = image_path_from_row(row)
        t0 = time.time()
        pred = predict_fn(path)
        elapsed = time.time() - t0

        pred["gt"] = row["gt"]
        pred["model"] = model_name
        results.append(pred)

        print(f"[{i+1}/{len(sample)}] {elapsed:.1f}s — pred={pred.get('predicted_class')} gt={row['gt']}")

        if checkpoint_path:
            pd.DataFrame(results).to_json(checkpoint_path, orient="records", indent=2, force_ascii=False)

        if (i + 1) % 5 == 0:
            gc.collect()
            torch.cuda.empty_cache()
            try:
                torch._dynamo.reset()
            except Exception:
                pass

    total = time.time() - start_time
    print(f"\n⏱️ Temps total : {total/60:.1f} min ({total/len(sample):.1f}s/image en moyenne)")

    df_results = pd.DataFrame(results)
    y_true = df_results["gt"].tolist()
    y_pred = df_results["predicted_class"].tolist()
    labels = ["normal", "suspected_opacity", "uncertain"]

    metrics = {
        "model": model_name,
        "n": len(df_results),
        "accuracy":  round(accuracy_score(y_true, y_pred), 4),
        "f1_macro":  round(f1_score(y_true, y_pred, labels=labels, average="macro",  zero_division=0), 4),
        "f1_weighted": round(f1_score(y_true, y_pred, labels=labels, average="weighted", zero_division=0), 4),
        "recall_opacity": round(recall_score(y_true, y_pred, labels=["suspected_opacity"], average="micro", zero_division=0), 4),
        "precision_opacity": round(precision_score(y_true, y_pred, labels=["suspected_opacity"], average="micro", zero_division=0), 4),
        "uncertain_rate": round((df_results["predicted_class"] == "uncertain").mean(), 4),
        "coverage": round((df_results["predicted_class"] != "uncertain").mean(), 4),
        "json_valid_pct":        round(df_results["raw"].apply(lambda r: bool(re.search(r"\{.*\}", str(r), re.DOTALL))).mean(), 4),
        "has_justification_pct": round(df_results.get("justification", pd.Series()).apply(lambda x: bool(x and len(str(x)) > 20)).mean(), 4),
        "has_warning_pct":       round(df_results.get("warning", pd.Series()).apply(lambda x: bool(x)).mean(), 4),
        "guardrail_rate":        round(df_results.get("guardrails_triggered", pd.Series()).apply(lambda x: len(x) > 0 if isinstance(x, list) else False).mean(), 4),
        "avg_confidence":        round(df_results["confidence"].mean(), 4),
    }
    return metrics, df_results



print("Évaluation du modèle fine-tuné sur le jeu de test complet...")
metrics_finetuned, df_finetuned = evaluate_model(
    predict_finetuned, df_test, "fine-tuné LoRA",
    checkpoint_path="checkpoint_finetuned.json"
)
print("✅ Terminé.")
print(json.dumps(metrics_finetuned, indent=2))


sample_30 = df_finetuned.head(30) if len(df_finetuned) >= 30 else df_finetuned

outputs_30 = []
for i, (_, row) in enumerate(sample_30.iterrows()):
    output = {
        "id": i + 1,
        "image_file": row.get("image_file", ""),
        "ground_truth": row["gt"],
        "model": "medgemma-4b-it + LoRA",
        "image_quality": row.get("image_quality"),
        "predicted_class": row.get("predicted_class"),
        "confidence": row.get("confidence"),
        "visual_evidence": row.get("visual_evidence", []),
        "justification": row.get("justification", ""),
        "limitations": row.get("limitations", []),
        "warning": row.get("warning", ""),
        "guardrails_triggered": row.get("guardrails_triggered", []),
        "correct": row.get("predicted_class") == row["gt"],
    }
    outputs_30.append(output)

with open("30_model_outputs.json", "w", encoding="utf-8") as f:
    json.dump(outputs_30, f, indent=2, ensure_ascii=False)

print(f"\n30 outputs sauvegardés dans 30_model_outputs.json")
print(f"   Accuracy sur ces 30 : {sum(o['correct'] for o in outputs_30)/len(outputs_30):.2%}")